# 中医病历多标签分类 — 最终版

**任务**：根据「症状规范 + YOLO标签」预测「证型规范」（多标签分类）

**特征**：症状规范（前200高频症状）+ YOLO标签（共6种）

**标签**：取频次前10的证型

**模型**：Logistic Regression（综合指标最优）

**流程**：先用 80/20 分割评估指标 → 再用全量数据重训练保存最终模型

## 1. 导入依赖

In [1]:
import pandas as pd
import numpy as np
import json
import pickle
import os
import warnings
warnings.filterwarnings('ignore')

from collections import Counter
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    hamming_loss, jaccard_score,
    f1_score, accuracy_score, roc_auc_score,
    classification_report
)

print('依赖导入完成')

依赖导入完成


## 2. 数据加载与清洗

In [2]:
df = pd.read_csv('data_final.csv')
print(f'原始数据形状: {df.shape}')

df = df[df['证型规范'].notna() & (df['证型规范'].str.strip() != '')].reset_index(drop=True)
print(f'过滤空证型后: {df.shape}')

def parse_list(s):
    if pd.isna(s) or str(s).strip() == '':
        return []
    return [x.strip() for x in str(s).split(',') if x.strip()]

df.head(3)

原始数据形状: (1562, 5)
过滤空证型后: (1562, 5)


,症状规范,中药规范,证型规范,舌象,YOLO标签
0,"咳嗽,喘息,头晕,汗出","人参,鹿茸,红景天,苍术,厚朴,李子,车前子,白附子,法半夏,紫菀,莪术,款冬花,金荞麦,冬...","脾肾亏虚,痰热壅肺","舌质暗,苔薄黄腻",red tongue yellow fur thick greasy fur
1,"下肢浮肿,汗出,纳差,面色晦暗","红景天,苍术,厚朴,李子,车前子,白附子,法半夏,紫菀,莪术,款冬花,金荞麦,冬瓜子,忍冬藤...","脾肾亏虚,痰热壅肺","舌质暗,苔薄白",The white tongue is thick and greasy
2,"头晕,纳差","人参,鹿茸,红景天,莱菔子,芥子,厚朴,大皂角,当归,莪术,紫菀,款冬花,法半夏,蚕沙,桑白...","脾肾亏虚,痰热壅肺",NaN,NaN


## 3. 确定特征集与标签集（基于全量数据统计）

> 统计频次必须用**全量数据**，保证前200症状/前10证型的定义与实际数据一致。

In [3]:
TOP_N_SYMPTOMS = 200
TOP_K_LABELS   = 10

# ── 前10证型 ──
zx_counter = Counter()
for ls in df['证型规范'].apply(parse_list): zx_counter.update(ls)
top_zhengxing = [z for z, _ in zx_counter.most_common(TOP_K_LABELS)]
print(f'前 {TOP_K_LABELS} 证型:')
for z, c in zx_counter.most_common(TOP_K_LABELS): print(f'  {z}: {c} 次')

# ── 前200症状 ──
sx_counter = Counter()
for ls in df['症状规范'].apply(parse_list): sx_counter.update(ls)
top_symptoms = [s for s, _ in sx_counter.most_common(TOP_N_SYMPTOMS)]
print(f'\n症状总种类: {len(sx_counter)}，取前 {TOP_N_SYMPTOMS}')
print(f'第200名症状: {top_symptoms[-1]}（频次: {sx_counter[top_symptoms[-1]]}）')

# ── YOLO 标签（全部保留）──
yolo_counter = Counter()
for y in df['YOLO标签'].dropna():
    y = str(y).strip()
    if y: yolo_counter[y] += 1
all_yolo_labels = sorted(yolo_counter.keys())
print(f'\nYOLO标签共 {len(all_yolo_labels)} 种: {all_yolo_labels}')

前 10 证型:
  痰热壅肺: 375 次
  痰浊阻肺: 368 次
  肺肾气虚: 258 次
  肺脾气虚: 188 次
  痰瘀内阻: 162 次
  肺肾气阴两虚: 151 次
  肺脾肾虚: 101 次
  风寒袭肺: 77 次
  外寒内饮: 77 次
  阳虚水泛: 53 次

症状总种类: 433，取前 200
第200名症状: 迂曲（频次: 1）

YOLO标签共 6 种: ['The red tongue is thick and greasy', 'The white tongue is thick and greasy', 'black tongue coating', 'map tongue coating', 'purple tongue coating', 'red tongue yellow fur thick greasy fur']


## 4. 构建全量特征矩阵与标签矩阵

In [4]:
# 过滤：只保留至少含1个top-10证型的样本
top_set = set(top_zhengxing)
df['labels_filtered'] = df['证型规范'].apply(
    lambda s: [l for l in parse_list(s) if l in top_set]
)
df_valid = df[df['labels_filtered'].apply(len) > 0].reset_index(drop=True)
print(f'有效样本数（含top-10证型）: {len(df_valid)}')

# 症状特征：固定 classes=top_symptoms，维度始终为 200
mlb_symptom = MultiLabelBinarizer(classes=top_symptoms)
X_sym = mlb_symptom.fit_transform(
    df_valid['症状规范'].apply(lambda s: [x for x in parse_list(s) if x in set(top_symptoms)])
)

# YOLO 特征：固定 classes=all_yolo_labels，维度始终为 6
mlb_yolo = MultiLabelBinarizer(classes=all_yolo_labels)
X_yolo = mlb_yolo.fit_transform(
    df_valid['YOLO标签'].apply(lambda y: [str(y).strip()] if pd.notna(y) and str(y).strip() else [])
)

# 合并：(N, 206)
X_all = np.hstack([X_sym, X_yolo])
print(f'特征矩阵: {X_all.shape}  (前{TOP_N_SYMPTOMS}症状 + {len(all_yolo_labels)} YOLO标签)')

# 标签矩阵
mlb_label = MultiLabelBinarizer(classes=top_zhengxing)
Y_all = mlb_label.fit_transform(df_valid['labels_filtered'])
print(f'标签矩阵: {Y_all.shape}  (前{TOP_K_LABELS}证型)')

有效样本数（含top-10证型）: 1386
特征矩阵: (1386, 206)  (前200症状 + 6 YOLO标签)
标签矩阵: (1386, 10)  (前10证型)


## 5. 划分训练集 / 测试集（用于评估）

In [5]:
X_train, X_test, Y_train, Y_test = train_test_split(
    X_all, Y_all, test_size=0.2, random_state=42
)
print(f'训练集: {X_train.shape}, 测试集: {X_test.shape}')

训练集: (1108, 206), 测试集: (278, 206)


## 6. 评估阶段：在 train/test 分割上训练并评估

In [6]:
def make_model():
    """返回一个新的、未训练的模型实例"""
    return OneVsRestClassifier(
        LogisticRegression(
            C=0.1,
            max_iter=1000,
            class_weight='balanced',  # 解决标签不平衡
            solver='lbfgs',
            random_state=42
        ),
        n_jobs=-1
    )

print('[1/2] 在 train/test 分割上训练（用于评估）...')
eval_model = make_model()
eval_model.fit(X_train, Y_train)
print('完成！')

[1/2] 在 train/test 分割上训练（用于评估）...


完成！


## 7. 模型评估（在 20% 测试集上）

In [7]:
Y_pred       = eval_model.predict(X_test)
Y_pred_proba = eval_model.predict_proba(X_test)

hl   = hamming_loss(Y_test, Y_pred)
acc  = accuracy_score(Y_test, Y_pred)
jacc = jaccard_score(Y_test, Y_pred, average='samples', zero_division=0)
f1mi = f1_score(Y_test, Y_pred, average='micro', zero_division=0)
f1ma = f1_score(Y_test, Y_pred, average='macro', zero_division=0)

try:
    auc_macro = roc_auc_score(Y_test, Y_pred_proba, average='macro')
except Exception as e:
    auc_macro = float('nan')
    print(f'[警告] Macro AUC 计算失败: {e}')

sep = '=' * 52
print(sep)
print('评估结果 — Logistic Regression (80%训练/20%测试)')
print(sep)
print(f'  Hamming Loss      : {hl:.4f}   ↓越低越好')
print(f'  Subset Accuracy   : {acc:.4f}   ↑越高越好')
print(f'  Jaccard Score     : {jacc:.4f}   ↑越高越好')
print(f'  F1-Score (Micro)  : {f1mi:.4f}   ↑越高越好')
print(f'  F1-Score (Macro)  : {f1ma:.4f}   ↑越高越好')
print(f'  Macro AUC         : {auc_macro:.4f}   ↑越高越好')
print(sep)
print('\n各证型详细分类报告:')
print(classification_report(Y_test, Y_pred, target_names=mlb_label.classes_, zero_division=0))

评估结果 — Logistic Regression (80%训练/20%测试)
  Hamming Loss      : 0.3129   ↓越低越好
  Subset Accuracy   : 0.0540   ↑越高越好
  Jaccard Score     : 0.2464   ↑越高越好
  F1-Score (Micro)  : 0.3235   ↑越高越好
  F1-Score (Macro)  : 0.2943   ↑越高越好
  Macro AUC         : 0.6857   ↑越高越好

各证型详细分类报告:
              precision    recall  f1-score   support

        痰热壅肺       0.53      0.77      0.63        73
        痰浊阻肺       0.28      0.44      0.34        71
        肺肾气虚       0.27      0.59      0.37        46
        肺脾气虚       0.24      0.57      0.33        51
        痰瘀内阻       0.18      0.55      0.28        31
      肺肾气阴两虚       0.16      0.50      0.25        28
        肺脾肾虚       0.12      0.44      0.19        27
        风寒袭肺       0.10      0.50      0.16        16
        外寒内饮       0.10      0.50      0.16        14
        阳虚水泛       0.13      0.78      0.23         9

   micro avg       0.23      0.57      0.32       366
   macro avg       0.21      0.56      0.29       366
weighted avg       0.

## 8. 全量重训练并保存最终模型

> **标准实践**：评估后用**全量数据**再训练一遍，模型见过更多样本，泛化能力更强。
> 保存的评估指标来自上一步的 hold-out 测试集，是对真实性能的客观估计。

In [8]:
# ── [2/2] 全量数据重训练 ──
print('[2/2] 用全量数据重训练最终模型（更完整）...')
final_model = make_model()
final_model.fit(X_all, Y_all)   # ← 全量，比评估阶段多 20% 样本
print(f'    训练样本数: {len(X_all)}（比评估阶段多 {len(X_all)-len(X_train)} 条）')
print('    重训练完成！')

# ── 保存 ──
SAVE_DIR = 'tcm_syndrome_model'
os.makedirs(SAVE_DIR, exist_ok=True)

with open(f'{SAVE_DIR}/model.pkl',       'wb') as f: pickle.dump(final_model, f)  # ← 全量模型
with open(f'{SAVE_DIR}/mlb_symptom.pkl', 'wb') as f: pickle.dump(mlb_symptom, f)
with open(f'{SAVE_DIR}/mlb_yolo.pkl',   'wb') as f: pickle.dump(mlb_yolo,    f)
with open(f'{SAVE_DIR}/mlb_label.pkl',  'wb') as f: pickle.dump(mlb_label,   f)

with open(f'{SAVE_DIR}/top_symptoms.json',  'w', encoding='utf-8') as f:
    json.dump(top_symptoms,    f, ensure_ascii=False, indent=2)
with open(f'{SAVE_DIR}/top_zhengxing.json', 'w', encoding='utf-8') as f:
    json.dump(top_zhengxing,  f, ensure_ascii=False, indent=2)
with open(f'{SAVE_DIR}/yolo_labels.json',   'w', encoding='utf-8') as f:
    json.dump(all_yolo_labels, f, ensure_ascii=False, indent=2)

meta = {
    'model_type': 'LogisticRegression (OneVsRest, C=0.1, class_weight=balanced)',
    'trained_on': 'full dataset',
    'n_samples_full': int(len(X_all)),
    'n_samples_eval_train': int(len(X_train)),
    'n_samples_eval_test': int(len(X_test)),
    'top_n_symptoms': TOP_N_SYMPTOMS,
    'top_k_labels': TOP_K_LABELS,
    'n_features': int(X_all.shape[1]),
    'n_labels': int(Y_all.shape[1]),
    'symptom_list': top_symptoms,
    'zhengxing_list': top_zhengxing,
    'yolo_label_list': all_yolo_labels,
    'eval_metrics': {
        'note': '以下指标来自 80/20 hold-out 评估，非全量训练集',
        'hamming_loss': round(hl, 4),
        'subset_accuracy': round(acc, 4),
        'jaccard': round(jacc, 4),
        'f1_micro': round(f1mi, 4),
        'f1_macro': round(f1ma, 4),
        'macro_auc': round(auc_macro, 4) if not np.isnan(auc_macro) else None
    }
}
with open(f'{SAVE_DIR}/meta.json', 'w', encoding='utf-8') as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print(f'\n✅ 所有文件已保存到: {os.path.abspath(SAVE_DIR)}')
for fname in sorted(os.listdir(SAVE_DIR)):
    size = os.path.getsize(f'{SAVE_DIR}/{fname}')
    print(f'  {fname:30s}  {size/1024:.1f} KB')

[2/2] 用全量数据重训练最终模型（更完整）...
    训练样本数: 1386（比评估阶段多 278 条）
    重训练完成！

✅ 所有文件已保存到: /Users/yangchenghao/PycharmProjects/graduation_project/machinelearning/tcm_syndrome_model
  meta.json                       4.5 KB
  mlb_label.pkl                   0.5 KB
  mlb_symptom.pkl                 3.9 KB
  mlb_yolo.pkl                    0.5 KB
  model.pkl                       21.0 KB
  top_symptoms.json               3.1 KB
  top_zhengxing.json              0.2 KB
  yolo_labels.json                0.2 KB


## 9. Django 调用示例

In [9]:
# ── 1. 加载模型（Django 应用启动时执行一次）──
import pickle, json, numpy as np

SAVE_DIR = 'tcm_syndrome_model'

with open(f'{SAVE_DIR}/model.pkl',       'rb') as f: _model       = pickle.load(f)
with open(f'{SAVE_DIR}/mlb_symptom.pkl', 'rb') as f: _mlb_symptom = pickle.load(f)
with open(f'{SAVE_DIR}/mlb_yolo.pkl',   'rb') as f: _mlb_yolo    = pickle.load(f)
with open(f'{SAVE_DIR}/mlb_label.pkl',  'rb') as f: _mlb_label   = pickle.load(f)
with open(f'{SAVE_DIR}/top_symptoms.json', encoding='utf-8') as f: _symptom_list = json.load(f)

# ── 2. 预测函数 ──
def predict_zhengxing(symptoms: list, yolo_label: str = '', threshold: float = 0.3):
    """
    symptoms   : list[str]  患者症状列表，元素从 top_symptoms 中选取
    yolo_label : str        舌象 YOLO 标签（可为空）
    threshold  : float      概率阈值，默认 0.3
    返回: list[dict]，每项 {'zhengxing': str, 'probability': float}，按概率降序
    """
    valid = [s for s in symptoms if s in set(_symptom_list)]
    X_sym  = _mlb_symptom.transform([valid])
    X_yolo = _mlb_yolo.transform([[yolo_label.strip()] if yolo_label.strip() else []])
    proba  = _model.predict_proba(np.hstack([X_sym, X_yolo]))[0]
    return sorted(
        [{'zhengxing': l, 'probability': round(float(p), 4)}
         for l, p in zip(_mlb_label.classes_, proba) if p >= threshold],
        key=lambda x: -x['probability']
    )

# ── 3. 测试 ──
result = predict_zhengxing(['咳嗽', '喘息', '痰色白', '胸闷', '乏力'],
                           'The white tongue is thick and greasy',
                           threshold=0.3)
print('预测结果（概率 >= 0.3）:')
for r in result:
    print(f"  {r['zhengxing']:12s}  {r['probability']:.4f}")

预测结果（概率 >= 0.3）:
  痰浊阻肺          0.6183
  肺肾气虚          0.6078
  风寒袭肺          0.5614
  痰瘀内阻          0.5521
  肺脾气虚          0.5324
  肺脾肾虚          0.4323
  外寒内饮          0.4231
  阳虚水泛          0.3922


---
## 附：Django 集成说明

### 文件清单
```
tcm_model/
├── model.pkl            # 全量数据训练的最终模型
├── mlb_symptom.pkl      # 症状编码器（固定200维）
├── mlb_yolo.pkl         # YOLO标签编码器（固定6维）
├── mlb_label.pkl        # 证型解码器
├── top_symptoms.json    # 前200症状列表（供前端患者勾选）
├── top_zhengxing.json   # 前10证型列表
├── yolo_labels.json     # 6种YOLO标签列表
└── meta.json            # 元信息 + hold-out 评估指标
```

### ml_service.py
```python
import pickle, json, numpy as np, os
from django.conf import settings

MODEL_DIR = os.path.join(settings.BASE_DIR, 'tcm_model')
_model       = pickle.load(open(f'{MODEL_DIR}/model.pkl',       'rb'))
_mlb_symptom = pickle.load(open(f'{MODEL_DIR}/mlb_symptom.pkl', 'rb'))
_mlb_yolo    = pickle.load(open(f'{MODEL_DIR}/mlb_yolo.pkl',    'rb'))
_mlb_label   = pickle.load(open(f'{MODEL_DIR}/mlb_label.pkl',   'rb'))
_symptom_list = json.load(open(f'{MODEL_DIR}/top_symptoms.json', encoding='utf-8'))

def predict_zhengxing(symptoms, yolo_label='', threshold=0.3):
    valid = [s for s in symptoms if s in set(_symptom_list)]
    X = np.hstack([
        _mlb_symptom.transform([valid]),
        _mlb_yolo.transform([[yolo_label.strip()] if yolo_label.strip() else []])
    ])
    proba = _model.predict_proba(X)[0]
    return sorted(
        [{'zhengxing': l, 'probability': round(float(p), 4)}
         for l, p in zip(_mlb_label.classes_, proba) if p >= threshold],
        key=lambda x: -x['probability']
    )
```

### POST /api/diagnose/ 请求
```json
{"symptoms": ["咳嗽", "喘息", "痰色白"], "yolo_label": "The white tongue is thick and greasy", "threshold": 0.3}
```